# RAG with Small Models — Ask an AI About the History of AI

This notebook builds a complete RAG (Retrieval Augmented Generation) pipeline using:
- **Knowledge base**: Wikipedia articles on the history of AI
- **Embedding model**: `all-mpnet-base-v2` (SentenceTransformers)
- **Vector database**: LanceDB (local, no server needed)
- **Model**: Qwen2.5-3B-Instruct in GGUF format (runs on CPU, no API key)

**The punchline**: We're asking an AI about the history of AI — using a small open-source model that costs nothing to run.

## Step 0 — Install Dependencies

⚠️ After running this cell, **restart the runtime** (`Runtime → Restart session`), then continue from Step 1.

In [ ]:
# Pin numpy FIRST — sentence-transformers will otherwise pull a conflicting version
!pip install numpy==1.26.4 --quiet

!pip install wikipedia-api sentence-transformers lancedb llama-cpp-python pyarrow pandas tqdm --quiet

print("Done. Restart the runtime before running any further cells.")

## Step 1 — Build the Knowledge Base

We pull Wikipedia articles on key moments and people in AI history.
This is our "document store" — the knowledge the model will retrieve from.

In [ ]:
import wikipediaapi

wiki = wikipediaapi.Wikipedia(user_agent='RAG-Demo/1.0', language='en')

article_titles = [
    "Alan Turing",
    "History of artificial intelligence",
    "Transformer (deep learning architecture)",
    "GPT-3",
    "AlphaGo",
    "ImageNet",
    "Generative adversarial network",
    "BERT (language model)",
    "Reinforcement learning from human feedback",
    "Large language model",
]

documents = []
for title in article_titles:
    page = wiki.page(title)
    if page.exists():
        documents.append({"title": title, "text": page.text})
        print(f"✓ {title} — {len(page.text):,} characters")
    else:
        print(f"✗ {title} — not found")

print(f"\nTotal articles loaded: {len(documents)}")

## Step 2 — Chunk the Text

RAG doesn't retrieve entire documents — it retrieves small chunks.
Why? Because embeddings work best on focused, specific passages.
We split each article into chunks of ~5 sentences.

In [ ]:
import re
from tqdm.auto import tqdm

CHUNK_SIZE = 5  # sentences per chunk
MIN_TOKENS = 30  # filter out tiny chunks

def split_into_sentences(text):
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    return [s.strip() for s in sentences if len(s.strip()) > 10]

def make_chunks(sentences, chunk_size):
    return [
        " ".join(sentences[i:i+chunk_size])
        for i in range(0, len(sentences), chunk_size)
    ]

all_chunks = []
for doc in tqdm(documents):
    sentences = split_into_sentences(doc["text"])
    chunks = make_chunks(sentences, CHUNK_SIZE)
    for chunk in chunks:
        token_count = len(chunk) / 4  # rough estimate: 1 token ≈ 4 chars
        if token_count >= MIN_TOKENS:
            all_chunks.append({
                "source": doc["title"],
                "text": chunk,
                "token_count": round(token_count, 1)
            })

print(f"Total chunks: {len(all_chunks)}")
print(f"\nSample chunk from '{all_chunks[0]['source']}':")
print(all_chunks[0]['text'][:300], "...")

## Step 3 — Embed the Chunks

Each chunk gets converted into a vector (768 numbers).
Similar meaning = similar vectors = close together in vector space.
This is what makes semantic search work.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedding_model = SentenceTransformer("all-mpnet-base-v2", device="cpu")

print("Embedding chunks on CPU...")
texts = [chunk["text"] for chunk in all_chunks]
embeddings = embedding_model.encode(texts, batch_size=32, show_progress_bar=True)

for i, chunk in enumerate(all_chunks):
    chunk["embedding"] = embeddings[i]

print(f"\nEmbedding shape: {embeddings.shape}")
print(f"Each chunk → {embeddings.shape[1]}-dimensional vector")

## Step 4 — Store in LanceDB

LanceDB is a local vector database — no server, no Docker, no cost.
It stores our chunks + embeddings and lets us do fast cosine similarity search.

In [ ]:
import lancedb
import pyarrow as pa

db = lancedb.connect("ai_history_db")

data = [
    {
        "source": chunk["source"],
        "text": chunk["text"],
        "token_count": float(chunk["token_count"]),
        "embedding": chunk["embedding"].astype(np.float32).tolist(),
    }
    for chunk in all_chunks
]

schema = pa.schema([
    pa.field("source", pa.string()),
    pa.field("text", pa.string()),
    pa.field("token_count", pa.float64()),
    pa.field("embedding", pa.list_(pa.float32(), list_size=768)),
])

table = db.create_table("ai_history", data=data, schema=schema, mode="overwrite")
table.create_index(metric="cosine", vector_column_name="embedding", index_type="IVF_FLAT")

print(f"Stored {len(data)} chunks in LanceDB")
print(f"Database saved at: ./ai_history_db")

## Step 5 — Test Retrieval

Before we add the model, let's verify retrieval works.
Ask a question — the system finds the most relevant chunks using vector similarity.
No language model involved yet. Just math.

In [ ]:
import textwrap

def retrieve(query, top_k=3):
    query_embedding = embedding_model.encode(query, convert_to_tensor=False)
    results = (
        db["ai_history"]
        .search(query_embedding, vector_column_name="embedding")
        .limit(top_k)
        .to_list()
    )
    return results

def show_results(query, results):
    print(f"Query: '{query}'\n")
    print("-" * 60)
    for i, r in enumerate(results):
        print(f"[{i+1}] Source: {r['source']}  |  Distance: {r['_distance']:.4f}")
        print(textwrap.fill(r['text'][:300] + "...", 80))
        print()

query = "Who invented the transformer architecture?"
results = retrieve(query)
show_results(query, results)

In [ ]:
query = "When did AlphaGo defeat Lee Sedol?"
results = retrieve(query)
show_results(query, results)

In [ ]:
query = "What did Alan Turing contribute to artificial intelligence?"
results = retrieve(query)
show_results(query, results)

## Step 6 — Load Qwen2.5-3B-Instruct (GGUF)

Qwen2.5-3B-Instruct in 4-bit GGUF format:
- 3 billion parameters
- Runs on CPU — no GPU needed
- No API key, no cost per token
- MIT license — fully open source

In [ ]:
from llama_cpp import Llama

llm = Llama.from_pretrained(
    repo_id="Qwen/Qwen2.5-3B-Instruct-GGUF",
    filename="*q4_k_m*",
    verbose=False,
    n_ctx=4096,
)

print("Model loaded. Ready to generate.")

## Step 7 — Build the RAG Pipeline

Retrieve relevant chunks → format them into a prompt → generate an answer.
This is the complete RAG loop.

In [ ]:
def format_prompt(query, context_chunks):
    context = "\n\n".join(
        [f"[Source: {c['source']}]\n{c['text']}" for c in context_chunks]
    )
    return f"""<|im_start|>system
You are a helpful AI historian. Answer questions based only on the provided context.
Be concise, accurate, and cite the source when relevant.<|im_end|>
<|im_start|>user
Context:
{context}

Question: {query}<|im_end|>
<|im_start|>assistant
"""


def ask(query, top_k=3, max_tokens=512):
    print(f"Question: {query}")
    print("=" * 60)

    chunks = retrieve(query, top_k=top_k)
    prompt = format_prompt(query, chunks)

    output = llm(
        prompt,
        max_tokens=max_tokens,
        stop=["<|im_end|>", "<|im_start|>"],
        echo=False,
    )

    answer = output["choices"][0]["text"].strip()
    print(f"\nAnswer:\n{answer}")
    print(f"\nSources used: {list(set(c['source'] for c in chunks))}")
    return answer

## Step 8 — Ask Questions

We're asking an AI — running locally, for free — about the history of AI.

Note: on CPU, each answer takes 1-2 minutes. That's expected — we'll fix that in the GPU version.

In [ ]:
ask("Who invented the transformer architecture and when?")

In [ ]:
ask("When did AlphaGo defeat Lee Sedol and why was it significant?")

In [ ]:
ask("What is RLHF and how is it used to train language models?")

In [ ]:
ask("What was Alan Turing's contribution to artificial intelligence?")

In [ ]:
ask("What is the difference between BERT and GPT?")

## CPU Benchmark — How Fast Is This?

Let's put a number on the speed. One question, timed end-to-end.

In [ ]:
import time

benchmark_query = "What was Alan Turing's contribution to artificial intelligence?"

print("Running on CPU...")
print("=" * 60)

chunks = retrieve(benchmark_query, top_k=3)
prompt = format_prompt(benchmark_query, chunks)

start = time.time()
output = llm(prompt, max_tokens=300, stop=["<|im_end|>", "<|im_start|>"], echo=False)
cpu_time = time.time() - start

answer = output["choices"][0]["text"].strip()
tokens_generated = len(answer.split())

print(f"Answer:\n{answer}")
print(f"\n--- CPU Stats ---")
print(f"Time taken:        {cpu_time:.1f} seconds")
print(f"Tokens generated:  ~{tokens_generated}")
print(f"Speed:             ~{tokens_generated/cpu_time:.1f} tokens/second")
print(f"\n>>> Switch to the GPU notebook to see what 20x faster looks like <<<")

## Bonus — What Happens Without RAG?

Ask the model the same question without any context.
This shows why RAG matters — the model can hallucinate or give vague answers
when it has no retrieval backing it up.

In [ ]:
def ask_without_rag(query, max_tokens=256):
    print(f"Question (no RAG): {query}")
    print("=" * 60)

    prompt = f"""<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
{query}<|im_end|>
<|im_start|>assistant
"""
    output = llm(
        prompt,
        max_tokens=max_tokens,
        stop=["<|im_end|>", "<|im_start|>"],
        echo=False,
    )
    answer = output["choices"][0]["text"].strip()
    print(f"\nAnswer:\n{answer}")
    return answer


query = "What is RLHF and how is it used to train language models?"

print("--- WITHOUT RAG ---")
ask_without_rag(query)

print("\n--- WITH RAG ---")
ask(query)